# EMNLP 2025 Corpus Mapping — Step 1: Audit and Cleaning

This notebook audits and cleans the EMNLP 2025 paper dataset before semantic embedding, clustering, and research-gap analysis.

Expected input file:
`data/emnlp_2025_main.xlsx`


In [ ]:
import pandas as pd
from pathlib import Path

# File path
DATA_FILE = Path("../data/emnlp_2025_main.xlsx")

# Load dataset
df = pd.read_excel(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()


## 1. Inspect dataset structure

Check the available columns, data types, and missing values.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())


## 2. Check for duplicate papers

Duplicate paper IDs or titles may indicate repeated records.

In [ ]:
print("Duplicate paper IDs:", df["paper_id"].duplicated().sum())
print("Duplicate titles:", df["title"].duplicated().sum())


## 3. Detect missing or empty abstracts

Some spreadsheet cells may contain blank strings rather than true missing values.

In [ ]:
df["abstract"] = df["abstract"].replace(r"^\s*$", pd.NA, regex=True)

print("Missing abstracts:", df["abstract"].isna().sum())


## 4. Inspect abstract lengths

Very short abstracts can indicate incomplete scraping or malformed records.

In [ ]:
df["abstract_word_count"] = (
    df["abstract"]
    .fillna("")
    .str.split()
    .str.len()
)

df["abstract_word_count"].describe()


In [ ]:
df[df["abstract_word_count"] < 20][
    ["paper_id", "title", "abstract", "abstract_word_count"]
].head(20)


## 5. Create combined text field

This combines each paper's title and abstract. The resulting `text` column will be used later for semantic embeddings.

In [ ]:
df["text"] = (
    df["title"].fillna("").str.strip()
    + ". "
    + df["abstract"].fillna("").str.strip()
)

df[["paper_id", "title", "text"]].head()


## 6. Create the cleaned corpus

For the initial corpus map, keep only papers with a paper ID, title, and abstract, and remove duplicate paper IDs.

In [ ]:
df_clean = (
    df
    .drop_duplicates(subset="paper_id")
    .dropna(subset=["paper_id", "title", "abstract"])
    .copy()
)

print("Original papers:", len(df))
print("Usable papers:", len(df_clean))
print("Removed:", len(df) - len(df_clean))


## 7. Save cleaned dataset

The cleaned CSV becomes the input for Step 2: semantic embeddings and clustering.

In [ ]:
OUTPUT_FILE = Path("../data/processed/emnlp_2025_clean.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_FILE, index=False)

print("Saved to:", OUTPUT_FILE)


## 8. Final audit summary

Run this cell and keep the output for your project notes or README.

In [ ]:
print("=== CORPUS AUDIT SUMMARY ===")
print(f"Original rows: {len(df)}")
print(f"Clean rows: {len(df_clean)}")
print(f"Removed rows: {len(df) - len(df_clean)}")
print(f"Duplicate paper IDs: {df['paper_id'].duplicated().sum()}")
print(f"Duplicate titles: {df['title'].duplicated().sum()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
print(f"Median abstract length: {df['abstract_word_count'].median():.0f} words")
